# Table of Contents

1. [Load Project](#Load-Project)
    1. [Load the HMM results](#Load-the-HMM-results)
2. [Plot 1: Heatmap Representations of HMM States](#Plot-1:-Heatmap-Representations-of-HMM-States)
    1. [Heatmap Representations of all genotypes](#Heatmap-representations-of-all-genotypes)
    2. [Interactive Heatmap version](#Interactive-Heatmap-version)
    3. [Interactive Heatmap version for all genotypes](#Interactive-Heatmap-version-of-all-genotypes)
3. [Plot 2: Probability of Surveillance across Node Types and Regions](#Plot-2:-Probability-of-Surveillance-across-Node-Types-and-Regions)
4. [Plot 3: Probability of States over Times](#Plot-3:-Probability-of-States-over-Time)
5. [Plot 4: Surveillance Probability by Bout Type](#Plot-4:-Surveillance-Probability-by-Bout-Type)

In [ ]:
# autoreload
%load_ext autoreload
%autoreload 2

# Load Project

In [ ]:
from pathlib import Path
import pandas as pd
from compass_labyrinth import load_project


project_path = Path(r'D:\Gladstone Dropbox\Palop Lab\Nina\Experiments\For Level 1 CoMPASS States\New_Reward_E3E4NLF\compass_analysis')

# Import config and metadata
config, cohort_metadata = load_project(project_path=project_path)

### Load the HMM results

In [ ]:
df_hmm = pd.read_csv(project_path / "results" / "compass_level_1" / "data_with_states.csv")
df_hmm

### State Representation: Characteristics

- State 1 --> Low Step Length, High Turn Angle --> "Surveillance" state (Red)
- State 2 --> High Step Length, Low Turn Angle --> "Ambulatory" state (Blue)

# Plot 1: Heatmap Representations of HMM States

This workflow visualizes the spatial distribution of HMM state proportions over a grid-mapped maze using heatmaps. It overlays key regions such as decision and target nodes for the specified genotype.

### Recommended Use:
1. Ensure `df_hmm` contains columns: 'Genotype', 'Grid.Number', 'HMM_State', 'x', and 'y'.
2. Set `genotype_name` to the genotype of interest (e.g., 'WT-WT').
3. Set `grid_filename` to the corresponding shapefile for the session.
4. Use `compute_state_proportion()` to calculate per-grid HMM state proportions.
5. Load the grid geometry using `create_grid_geodata()`.
6. Use `map_points_to_grid()` and `sjoin()` to align state estimates with grid polygons.
7. Merge state proportions back into the grid using `merge_state_proportions_to_grid()`.
8. Plot the heatmap using `plot_grid_heatmap()` and highlight decision/target regions.

In [ ]:
from compass_labyrinth.post_hoc_analysis.level_1 import (
    compute_state_proportion,
    create_grid_geodata,
    map_points_to_grid,
    merge_state_proportions_to_grid,
    plot_grid_heatmap,
)

genotype_name = 'E3'
grid_filename = "Session0001_grid.shp"
HMM_STATE = 1  # Fast / Ambulatory

# Step 1: Compute HMM state proportions by grid
state_df = compute_state_proportion(
    df=df_hmm,
    genotype_name=genotype_name,
    hmm_state=HMM_STATE,
)

# Step 2: Load session-specific grid geometry
grid = create_grid_geodata(
    config=config,
    grid_filename=grid_filename,
)

# Step 3: Map mean (x, y) points to grid polygons
pointInPolys = map_points_to_grid(state_df, grid)

# Step 4: Merge proportions with grid polygons
grid_mapped = merge_state_proportions_to_grid(grid, state_df)

# Step 5: Plot heatmap
# highlight_grids dropped -- this maze has no branching, so there is no "decision"
# node type to highlight (NODE_TYPE_MAPPING has no such key for this maze)
ax = plot_grid_heatmap(
    config=config,
    grid=grid_mapped,
    genotype_name=genotype_name,
    hmm_state=HMM_STATE,
    highlight_grids=None,
    target_grids="target_zone",
)

print('Target Nodes highlighted in YELLOW')

### Stationary state heatmap (hmm_state = 0)

Unlike the original two-state (Surveillance / Ambulatory) design, this dataset has a third category -- frames excluded from HMM fitting because their step length was too small to reliably estimate a turning angle (see notebook 04). Worth its own spatial heatmap rather than leaving it out, since `compute_state_proportion` etc. are already generic to `hmm_state`.

In [ ]:
# Step 1: Compute proportions for the stationary state (hmm_state=0)
state_df_stationary = compute_state_proportion(
    df=df_hmm,
    genotype_name=genotype_name,
    hmm_state=0,
)

# Step 2: Merge into the same grid geometry loaded above
grid_mapped_stationary = merge_state_proportions_to_grid(grid, state_df_stationary)

# Step 3: Plot heatmap
ax = plot_grid_heatmap(
    config=config,
    grid=grid_mapped_stationary,
    genotype_name=genotype_name,
    hmm_state=0,
    highlight_grids=None,
    target_grids="target_zone",
)

print('Target Nodes highlighted in YELLOW')

## Heatmap representations of all genotypes

In [ ]:
from compass_labyrinth.post_hoc_analysis.level_1 import plot_all_genotype_heatmaps


plot_all_genotype_heatmaps(
    config=config,
    df_hmm=df_hmm,
    grid_filename=grid_filename,
    highlight_grids=None,
    target_grids="target_zone",
    hmm_state=0,
    cmap='RdBu',
)

### Heatmap representations of all genotypes -- Stationary state

In [ ]:
plot_all_genotype_heatmaps(
    config=config,
    df_hmm=df_hmm,
    grid_filename=grid_filename,
    highlight_grids=None,
    target_grids="target_zone",
    hmm_state=0,
    cmap='RdBu',
)

## Interactive Heatmap version

In [ ]:
from compass_labyrinth.post_hoc_analysis.level_1 import (
    compute_state_proportion,
    create_grid_geodata,
    get_grid_centroids,
    plot_interactive_heatmap,
    overlay_trajectory_lines_plotly,
)


genotype_name = 'WT'

# 1. Compute state proportions
state_df = compute_state_proportion(df_hmm, genotype_name)

# 2. Load session-specific grid geometry
grid = create_grid_geodata(
    config=config,
    grid_filename=grid_filename,
)

# 3. Merge proportion values to grid
grid_mapped = merge_state_proportions_to_grid(grid, state_df)

# 4. Get grid centroids
grid_centroids = get_grid_centroids(grid_mapped)

# 5. Plot heatmap
fig = plot_interactive_heatmap(
    config=config,
    grid_mapped=grid_mapped,
    genotype_name=genotype_name,
    decision_grids="decision_reward",
    target_grids="target_zone",
    show_fig=False,
    return_fig=True,
)

# 6. Overlay smooth trajectory lines
overlay_trajectory_lines_plotly(
    fig=fig,
    df_hmm=df_hmm,
    genotype_name=genotype_name,
    grid_centroids=grid_centroids,
    top_percent=.1,
)

# 7. Show
fig.show()

## Interactive Heatmap version of all genotypes

In [ ]:
from compass_labyrinth.post_hoc_analysis.level_1 import plot_all_genotype_interactive_heatmaps


plot_all_genotype_interactive_heatmaps(
    config=config,
    df_hmm=df_hmm,
    grid_filename="Session-3 grid.shp",
    hmm_state=2,
    decision_grids="decision_reward",
    target_grids="target_zone",
    top_percent=.1,
)

# Plot 2: Probability of Surveillance across Node Types and Regions

This workflow computes and visualizes the probability of HMM state occupancy across behavioral Node Types/Regions

### Recommended Use:
1. Ensure `df_hmm` includes 'Genotype', 'Session', 'HMM_State', 'NodeType', and 'Grid.Number'.
2. If analyzing decision complexity, pass lists of `decision_3way` and `decision_4way` grid numbers.
3. Use `compute_state_probability()` to get HMM state proportions per category.
4. Use `plot_state_probability_boxplot()` to visualize across genotypes.

In [ ]:
from compass_labyrinth.post_hoc_analysis.level_1 import (
    compute_state_probability,
    plot_state_probability_boxplot,
)


column_of_interest = 'node_type'
values_displayed = [
    'Entry Zone', 'Straight (Reward Path)', 'Corner (Reward Path)',
    'Target Zone', 'Straight (Overshoot)', 'Corner (Overshoot)',
]
state = 1  # Slow / Surveillance

# Step 1: Compute proportions
state_count_df = compute_state_probability(
    df_hmm=df_hmm,
    column_of_interest=column_of_interest,
    values_displayed=values_displayed,
    state=state,
)

# Step 2: Plot boxplot
plot_state_probability_boxplot(
    config=config,
    state_count_df=state_count_df,
    column_of_interest=column_of_interest,
    state=state,
)

In [ ]:
from compass_labyrinth.post_hoc_analysis.level_1 import run_pairwise_ttests


# T-tests amongst genotypes per category
ttest_results = run_pairwise_ttests(state_count_df, column_of_interest='node_type')
print(ttest_results.sort_values("p_value"))

# Plot 3: Probability of States over Time

This workflow analyzes how the probability of being in a specific HMM state (e.g., State 2) evolves over time at decision vs. non-decision nodes. It computes median probabilities across sessions using sliding time bins and plots the resulting curves to compare decision-related dynamics.

### Recommended Use:
1. Ensure `df_hmm` contains columns: 'Time', 'Session', 'Grid.Number', 'HMM_State', 'NodeType', and 'Genotype'.
2. Define grid numbers corresponding to `Decision_Reward` and `NonDecision_Reward` nodes.
3. Set a time window (`lower_limit` to `upper_limit`) and a `bin_size` (e.g., 2000 frames) to compute time bins.
4. Use `compute_node_state_medians_over_time()` to calculate median state occupancy at decision vs. non-decision nodes.
5. Optionally filter the resulting DataFrame (`Deci_DF`) using a `threshold` to only keep early time bins.
6. Visualize the trajectory of state occupancy using `plot_node_state_median_curve()`.

In [ ]:
from compass_labyrinth.post_hoc_analysis.level_1 import (
    get_max_session_row_bracket,
    get_min_session_row_bracket,
    compute_node_state_medians_over_time,
    plot_node_state_median_curve,
)


lower_limit = 0
upper_limit = get_max_session_row_bracket(df_hmm)
threshold =  get_min_session_row_bracket(df_hmm)  # Only show bins where all sessions are present
bin_size = 2000
# 4 genotypes x 2 node types = 8 categories -- a 2-color list left most of them
# indistinguishable, so use a proper qualitative palette instead
palette = 'tab10'
# ylimit widened from (0.6, 1.1) -> (0.0, 1.1): Corner (Reward Path)'s Ambulatory
# state probability clusters around 0.25 (mice slow down at turns, per Plot 2),
# while Straight (Reward Path) clusters near 1.0 -- (0.6, 1.1) clipped every
# Corner line out of view entirely
figure_ylimit = (0.0, 1.1)

# This maze has no decision nodes -- substituting the geometrically meaningful
# analog confirmed in Plot 2: corners (turns) vs straight corridor, both on the
# reward path.
deci_df = compute_node_state_medians_over_time(
    df_hmm=df_hmm,
    state_types=[1],
    lower_lim=lower_limit,
    upper_lim=upper_limit,
    bin_size=bin_size,
    zone_names=["corner_reward", "nondecision_reward"],
    zone_labels={"corner_reward": "Corner (Reward Path)", "nondecision_reward": "Straight (Reward Path)"},
)

# Step 2: Optional filter to only plot early session bins
deci_df = deci_df.loc[deci_df.time_bins < threshold]

# Step 3: Plot time-evolving median probability curves
plot_node_state_median_curve(
    config=config,
    deci_df=deci_df,
    palette=palette,
    figure_ylimit=figure_ylimit,
    fig_title='Median Probability of Ambulatory State: Corner vs Straight (Reward Path)'
)

# Plot 4: Surveillance Probability by Bout Type

This workflow evaluates behavioral surveillance patterns at decision nodes across navigational bouts in the maze. It segments the session into bouts, computes  surveillance probability for each bout at decision nodes, and visualizes the average surveillance behavior across successful and unsuccessful bouts.

### Recommended Use:
1. Ensure `df_hmm` contains columns: 'Grid.Number', 'Session', 'Genotype', and maze node annotations.
2. Use `assign_bout_indices()` to segment sessions into bouts based on re-entries into the maze (e.g., delimiter node 47).
3. Run `compute_surveillance_probabilities()` to calculate surveillance at specified decision nodes for each bout.
4. Use `plot_surveillance_by_bout()` to generate a grouped barplot comparing surveillance probability between successful and unsuccessful bouts, including a t-test p-value.

In [ ]:
from compass_labyrinth.behavior.behavior_metrics.task_performance_analysis import assign_bout_indices_from_entry_zone
from compass_labyrinth.post_hoc_analysis.level_1 import (
    compute_surveillance_probabilities,
    plot_surveillance_by_bout,
)
from compass_labyrinth.constants import ENTRY_ZONE


# Assign Bout Numbers
# Bout = Entry zone --> Other non-entry nodes --> Entry zone
# Reusing the already-fixed entry-zone-based segmentation from
# task_performance_analysis: a single delimiter node (e.g. 0) collapses 5
# sessions into one giant bout because sparse tracking never touches that
# exact cell for a whole session -- delimiter_nodes=ENTRY_ZONE ([0, 1]) fixes it.
df_hmm = assign_bout_indices_from_entry_zone(
    df_hmm,
    delimiter_nodes=ENTRY_ZONE,
)

# Compute surveillance probability at Corner (Reward Path) nodes by Bout type
# This maze has no decision nodes -- substituting corner_reward (turns), the
# same geometrically meaningful analog used in Plots 2 and 3.
# Successful-> reached the target atleast once/Unsuccessful-> doesn't reached the target
index_df, median_df = compute_surveillance_probabilities(
    df_hmm=df_hmm,
    decision_nodes="corner_reward",
)

# Barplot to depict the above with ttest-ind pvalue
# ylim raised from 0.6 -> 0.85: real probability_1 values here range ~0.51-0.74,
# so 0.6 was clipping nearly every bar flat at the ceiling
plot_surveillance_by_bout(
    config=config,
    median_df=median_df,
    ylim=0.85,
)

In [ ]:
from compass_labyrinth.post_hoc_analysis.level_1 import run_within_genotype_mixedlm_with_fdr


# LMM for same genotype comparison across Bout types
df_within = run_within_genotype_mixedlm_with_fdr(index_df)

# Print results
print("Within-Genotype (Successful vs Unsuccessful):")
print(df_within)

In [ ]:
from compass_labyrinth.post_hoc_analysis.level_1 import test_across_genotypes_per_bout


# T-test across genotypes under Successful Bouts
df_across_success = test_across_genotypes_per_bout(median_df, bout_type='Successful')

# T-test across genotypes under Unsuccessful Bouts
df_across_unsuccess = test_across_genotypes_per_bout(median_df, bout_type='Unsuccessful')

# Print results
print("\n Across Genotypes (Successful only):")
print(df_across_success)

print("\n Across Genotypes (Unsuccessful only):")
print(df_across_unsuccess)